In [ ]:
from pathlib import Path
import shutil
import random
import cv2
import numpy as np
import torch
from PIL import Image, ImageOps
from tqdm.notebook import tqdm

from src.detection import GroundingDINODetector
from src.segmentation import SAMSegmenter
from src.data_utils import copy_coco_persons, get_image_hash

random.seed(42)

In [ ]:
SOURCE_DIR = Path("datasets/raw/google dated_converted_jpg/trashcans/trashcans_lens")
PROJECT_DIR = Path("datasets/ready/full_dataset")

# Pre-labeled humans
COCO_DATASET = Path("datasets/processed/coco_persons")
INCLUDE_COCO = False
MAX_COCO_IMAGES = None  # None = all

# Classes to label in SOURCE_DIR. Full set: {"red ball": 0, "human": 1, "trashcan": 2}
CLASS_MAP = {
    "trashcan": 2
}

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

source_count = len([p for p in SOURCE_DIR.rglob("*") if p.suffix.lower() not in ['.Identifier']])

print(f"Device: {DEVICE}")
print(f"Source: {SOURCE_DIR} ({source_count} files)")
print(f"Project: {PROJECT_DIR}")
print(f"COCO: {COCO_DATASET} (included: {INCLUDE_COCO})")

In [ ]:
# Zone.Identifier side files from Windows downloads
for identifier_file in SOURCE_DIR.rglob("*.Identifier"):
    identifier_file.unlink()

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif', '.tiff', '.svg', '.heic'}


def convert_to_jpg(source_folder: Path, output_folder: Path):
    """Convert every image under source_folder to RGB JPG, keeping the folder structure."""
    output_folder.mkdir(parents=True, exist_ok=True)
    converted_count = 0

    for file_path in tqdm(list(source_folder.rglob("*")), desc="Converting to JPG"):
        if ':Zone.Identifier' in str(file_path) or not file_path.is_file():
            continue
        if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        output_path = output_folder / file_path.relative_to(source_folder).parent / f"{file_path.stem}.jpg"
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with Image.open(file_path) as img:
            if img.mode in ('RGBA', 'LA', 'P'):
                rgb_img = Image.new('RGB', img.size, (255, 255, 255))
                if img.mode == 'P':
                    img = img.convert('RGBA')
                rgb_img.paste(img, mask=img.split()[-1])
                img = rgb_img
            elif img.mode != 'RGB':
                img = img.convert('RGB')

            img.save(output_path, 'JPEG', quality=95)
            converted_count += 1

    print(f"Converted {converted_count} images")

In [ ]:
CONVERT_TO_JPG = True

if CONVERT_TO_JPG:
    converted_source = SOURCE_DIR.parent / (SOURCE_DIR.name + "_converted_jpg")
    convert_to_jpg(SOURCE_DIR, converted_source)
    SOURCE_DIR = converted_source
    print(f"SOURCE_DIR is now {SOURCE_DIR}")

In [ ]:
BOX_THRESHOLD = 0.116
TEXT_THRESHOLD = 0.2
PROMPT = " . ".join(CLASS_MAP.keys())

detector = GroundingDINODetector(
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
    device=DEVICE
)
segmenter = SAMSegmenter()

In [ ]:
def mask_to_polygon(mask: np.ndarray) -> list[float]:
    """Largest contour of a binary mask, simplified, as normalized YOLO coordinates."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []

    c = max(contours, key=cv2.contourArea)
    approx = cv2.approxPolyDP(c, 0.005 * cv2.arcLength(c, True), True)
    if len(approx) < 3:
        return []

    h, w = mask.shape
    points = []
    for point in approx:
        x, y = point[0]
        points.extend([x / w, y / h])
    return points


def save_yolo_label(file_path: Path, labels: list[tuple[int, list[float]]]):
    with open(file_path, 'w') as f:
        for class_id, points in labels:
            f.write(f"{class_id} {' '.join(f'{p:.6f}' for p in points)}\n")

In [ ]:
# The validation set is copied once from fixed_val, never regenerated

VAL_DIR = PROJECT_DIR / "val"
VAL_IMAGES_DIR = VAL_DIR / "images"
VAL_LABELS_DIR = VAL_DIR / "labels"
FIXED_VAL_SOURCE = Path("datasets/ready/fixed_val")

if not VAL_IMAGES_DIR.exists() or not any(VAL_IMAGES_DIR.iterdir()):
    if FIXED_VAL_SOURCE.exists():
        VAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
        VAL_LABELS_DIR.mkdir(parents=True, exist_ok=True)

        for img_path in (FIXED_VAL_SOURCE / "images").glob("*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                shutil.copy(img_path, VAL_IMAGES_DIR / img_path.name)
        for lbl_path in (FIXED_VAL_SOURCE / "labels").glob("*.txt"):
            shutil.copy(lbl_path, VAL_LABELS_DIR / lbl_path.name)

        print(f"Validation set initialized from {FIXED_VAL_SOURCE}")
    else:
        print(f"{FIXED_VAL_SOURCE} does not exist, validation set is empty")
else:
    print(f"Keeping existing validation set ({len(list(VAL_IMAGES_DIR.glob('*')))} images)")

In [ ]:
existing_hashes = set()

for split in ['train', 'val', 'test']:
    img_dir = PROJECT_DIR / split / "images"
    if img_dir.exists():
        for img_path in img_dir.glob("*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                existing_hashes.add(get_image_hash(img_path))

print(f"Found {len(existing_hashes)} existing images in project")

In [ ]:
for split in ['train', 'test']:
    (PROJECT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (PROJECT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

source_images = []
if SOURCE_DIR.exists():
    source_images = [p for p in SOURCE_DIR.rglob("*") if p.suffix.lower() not in ['.Identifier']]
print(f"Found {len(source_images)} images in {SOURCE_DIR}")

added_count = 0
skipped_count = 0
error_count = 0

# GroundingDINO over-detects humans, they need a higher confidence
HUMAN_CONF_THRESHOLD = 0.5

for img_path in tqdm(source_images, desc="Processing Images"):
    try:
        img_hash = get_image_hash(img_path)
        if img_hash in existing_hashes:
            skipped_count += 1
            continue

        try:
            pil_img = ImageOps.exif_transpose(Image.open(img_path))
            if pil_img.mode != 'RGB':
                pil_img = pil_img.convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path.name}: {e}")
            error_count += 1
            continue

        detections = detector.detect(img_path, PROMPT, return_all_by_label=True)

        yolo_labels = []
        for label_name, dets in detections.items():
            if label_name not in CLASS_MAP:
                continue
            class_id = CLASS_MAP[label_name]

            for det in dets:
                if label_name == "human" and det['confidence'] < HUMAN_CONF_THRESHOLD:
                    continue

                masks_result = segmenter.segment_bbox(pil_img, det['bbox'])
                if masks_result and masks_result[0].masks is not None:
                    mask = masks_result[0].masks.data[0].cpu().numpy().astype(np.uint8)
                    points = mask_to_polygon(mask)
                    if points:
                        yolo_labels.append((class_id, points))

        # New images never go to val, it stays fixed
        split = 'train' if random.random() < 0.8 else 'test'

        pil_img.save(PROJECT_DIR / split / "images" / f"{img_hash}.jpg", quality=95)
        save_yolo_label(PROJECT_DIR / split / "labels" / f"{img_hash}.txt", yolo_labels)

        existing_hashes.add(img_hash)
        added_count += 1

    except Exception as e:
        print(f"Failed to process {img_path.name}: {e}")
        error_count += 1

print(f"Added {added_count}, skipped {skipped_count} duplicates, {error_count} errors")
print(f"Total in project: {len(existing_hashes)}")

In [ ]:
# COCO goes to train only, the validation set stays fixed
if INCLUDE_COCO:
    coco_count, existing_hashes = copy_coco_persons(
        coco_dir=COCO_DATASET,
        output_images=PROJECT_DIR / "train" / "images",
        output_labels=PROJECT_DIR / "train" / "labels",
        existing_hashes=existing_hashes,
        max_images=MAX_COCO_IMAGES,
        prefix="coco_"
    )
    print(f"Total in project after COCO: {len(existing_hashes)}")

In [ ]:
PREVIEW_DIR = PROJECT_DIR / "preview"
PREVIEW_DIR.mkdir(exist_ok=True)

COLORS = [(0, 0, 255), (0, 255, 0), (255, 0, 0)]  # BGR


def draw_yolo_labels(img_path, label_path, output_path):
    if not img_path.exists():
        return

    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if len(parts) < 2:
                continue

            class_id = int(parts[0])
            coords = [float(p) for p in parts[1:]]
            points = [[int(coords[i] * w), int(coords[i + 1] * h)] for i in range(0, len(coords), 2)]

            color = COLORS[class_id % len(COLORS)]
            cv2.polylines(img, [np.array(points, np.int32).reshape((-1, 1, 2))], True, color, 2)
            cv2.putText(img, f"Class {class_id}", (points[0][0], points[0][1] - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imwrite(str(output_path), img)


SAMPLES_PER_SPLIT = 5

for split in ['train', 'val', 'test']:
    img_dir = PROJECT_DIR / split / "images"
    lbl_dir = PROJECT_DIR / split / "labels"
    if not img_dir.exists():
        continue

    images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    for img_path in random.sample(images, min(len(images), SAMPLES_PER_SPLIT)):
        draw_yolo_labels(img_path, lbl_dir / f"{img_path.stem}.txt", PREVIEW_DIR / f"{split}_{img_path.name}")

In [ ]:
from utils import DisplayPath

DisplayPath(PREVIEW_DIR).display()